## Import packages

See YAML file for specific package requirements

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage.measure import regionprops, regionprops_table
from keras.utils import load_img
from keras.saving import load_model
from importlib import reload
import segmenteverygrain as seg
import sez
from segment_anything import sam_model_registry, SamPredictor
from tqdm import trange, tqdm
import os
import geopandas as gpd
import rasterio
%matplotlib qt
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
import rasterio
from shapely.geometry import Polygon
# create geopandas dataframe
import geopandas
import tifffile
from shapely import wkt
from rasterio.features import rasterize
# Assume you have your original image loaded to get shape:
import cv2

## Setting up and loading models/checkpoints

Insert model fname

In [ ]:
#path to where your segmenteveryzircon model is stored
model_fname = "/Users/omw339/Library/CloudStorage/OneDrive-TheUniversityofTexasatAustin/Desktop/segmenteverygrain/segmenteverygrain/zircons_2_26.keras"
# the SAM model checkpoints can be downloaded from: https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
checkpoint = "/Users/omw339/Downloads/sam_vit_h_4b8939.pth"
#path of the image you would like to segment
image_fname = '/Users/omw339/Desktop/OWBP25004/OWBP25004.tif'

load model and SAM checkpoints

In [ ]:
model = load_model(model_fname, custom_objects={'weighted_crossentropy': seg.weighted_crossentropy})
sam = sam_model_registry["default"](checkpoint=checkpoint)

## Run segmentation

Grains are supposed to be well defined in the image; e.g., if a grain consists of only a few pixels, it is unlikely to be detected.

The segmentation can take a few minutes even for medium-sized images. Images with ~2000 pixels along their largest dimension are a good start and allow the user to get an idea about how well the segmentation works.

If you have a much larger image, see the section **"Run segmentation on large image"** at the end of the notebook. Running the `predict_large_image` function takes a lot longer (e.g., several hours), but it is possible to analyze very large images with tens of thousands of grains.

In [ ]:
all_grains, image_pred, all_coords = seg.predict_large_image(image_fname, model, sam, min_area=400.0, patch_size=2000, overlap=200, remove_large_objects=True)

## Plot initial prediction image and Initialize grain list/labels

In [ ]:
image = np.array(load_img(image_fname))
fig, ax = plt.subplots()
all_grains, labels, pred_mask = seg.get_grains_from_patches(ax, image)
sez.plot_grains(image_fname, all_grains, step='Initial')

Initial number of segmented polygons

In [ ]:
initial_n = len(all_grains)
print(initial_n)

## Delete or merge grains in segmentation result
* click on the grain that you want to remove and press the 'x' key
* click on two grains that you want to merge and press the 'm' key (they have to be the last two grains you clicked on)
* press the 'g' key to hide the grain masks (so that you can see the original image better); press the 'g' key again to show the grain masks

In [ ]:
grain_inds = []
cid1 = fig.canvas.mpl_connect('button_press_event', 
                              lambda event: seg.onclick2(event, all_grains, grain_inds, ax=ax))
cid2 = fig.canvas.mpl_connect('key_press_event', 
                              lambda event: seg.onpress2(event, all_grains, grain_inds, fig=fig, ax=ax))

In [ ]:
fig.canvas.mpl_disconnect(cid1)
fig.canvas.mpl_disconnect(cid2)

Use this function to update the 'labels' array after deleting and merging grains (the 'all_grains' list is updated when doing the deletion and merging):

In [ ]:
all_grains, labels, gt_mask = seg.get_grains_from_patches(ax, image)

plot image after initial deletions

In [ ]:
image = np.array(load_img(image_fname))
fig, ax = plt.subplots()
sez.plot_grains(image_fname, all_grains, step='Deletions')

## Add new grains using the Segment Anything Model

* click on unsegmented grain that you want to add
* press the 'x' key if you want to delete the last grain you added
* press the 'm' key if you want to merge the last two grains that you added
* right click outside the grain (but inside the most recent mask) if you want to restrict the grain to a smaller mask - this adds a background prompt

In [ ]:
reload(seg)
predictor = SamPredictor(sam)
predictor.set_image(image) # this can take a while
coords = []
cid3 = fig.canvas.mpl_connect('button_press_event', lambda event: seg.onclick_large_image(event, ax, coords, image, predictor, patch_size=500))
cid4 = fig.canvas.mpl_connect('key_press_event', lambda event: seg.onpress(event, ax, fig))

In [ ]:
fig.canvas.mpl_disconnect(cid3)
fig.canvas.mpl_disconnect(cid4)

In [ ]:
all_grains, labels, gt_mask = seg.get_grains_from_patches(ax, image)

Plotting after additions and saving figure

In [ ]:
sez.plot_grains(image_fname, all_grains, step='Additions')

## Once you are happy with your segmentation results:

In [ ]:
all_grains, labels, gt_mask = seg.get_grains_from_patches(ax, image)

In [ ]:
final_n = len(all_grains)
print(final_n)

saving final figure of segmented polygons

In [ ]:
sez.plot_grains(image_fname, all_grains, step='Additions')

### Creating Metadata Table for Segmentation Results

In [ ]:
df_meta = sez.create_metadata_table(image_fname, final_n, model_fname, save_csv=False)

After you are done with the deletion / addition of grain masks, run this cell to generate an updated set of grains:

## Last Steps: Save mask, grain polygons, and grain coordinates
- These steps are vital if you would like to come back to your work later 
- The below command creates the mask associated with the image you just segmented. The line should return 'True' meaning that the file was created. You can double check this by going into the folder where you saved the file.

In [ ]:
#saving binary mask
cv2.imwrite('/Users/omw339/Desktop/summer_SEZ_outputs/OWBP25013/OWBP25013__mask_draft_9_21.png', gt_mask)


Last portion, run the following cells to create a geopandas geodataframe to store all of the polygons with their respective geometries and coordinates

In [ ]:
dataset = rasterio.open(image_fname)
projected_polys = []
for grain in all_grains:
    x, y = rasterio.transform.xy(
        dataset.transform, grain.exterior.xy[1], grain.exterior.xy[0]
    )
    poly = Polygon(np.vstack((x, y)).T)
    projected_polys.append(poly)
gdf = geopandas.GeoDataFrame(projected_polys, columns=["geometry"])
gdf.head()

saving the csv file with the grain coordinates

In [ ]:
gdf.to_csv('/Users/omw339/Desktop/summer_SEZ_outputs/OWBP25012/OWBP25012_final_10_15.csv')

## Steps to Re-Read your segmented grains back after finishing segmentation

### Step 1: Load polygons into geodataframe and re-initialize all_grains

In [ ]:
# Replace 'your_polygons.csv' with your actual CSV filename
csv_path = '/Users/omw339/Desktop/summer_SEZ_outputs/OWBP25001/OWBP25001_draft_coordinates_9_16_25_final.csv'

gdf = sez.load_polygons("path/to/your/file.csv", crs="EPSG:4326")
all_grains = list(gdf.geometry)

plot the loaded grains on the image

In [ ]:
sez.plot_grains(image_fname, all_grains, step='Initial')

now that all_grains is re-initialized and your grains are re-plotted, you are good to go back to adding and deleting grains!